# Huakun Huang | Frozen ELMo Small + TextCNN
Archived A/B/C/D results on the 591-row training pool. This notebook reads the contributor’s displayed tables and figures. See models/kun for extracted model and training sources.

## 1. Model and preprocessing
Frozen ELMo Small: **13,593,312 parameters**. Each complete classification head: **71,144 trainable parameters**. Five independently initialized heads (seeds 101–105) are averaged with equal 20% weights.

A learned scalar mixture of three ELMo layers is projected to 128 dimensions. Convolutions of widths 2/3/4, each with 32 channels, use both masked max and mean pooling. Their 192-dimensional representation is concatenated with a 128-dimensional global mean branch, then dropout (0.5) and a four-class linear classifier.

The design aims to combine local phrase cues with sentence-wide information while limiting trainable capacity. This study does not independently establish superiority over other encoders or benefits from explicit entity conditioning.

Tokenization uses `TweetTokenizer(preserve_case=True, reduce_len=True)` after encoding repair and URL/user normalization. Inputs are truncated to 64 tokens; empty inputs use `EMPTY`. ELMo `batch_to_ids` converts tokens to character IDs. All variants reuse the same frozen training features.

## 4. Fixed evaluation protocol
Only the **591 training rows** and their frozen features are used. Whitespace word-set Jaccard ≥ 0.5 defines connected groups (579 groups, 12 edges). Grouping is separate from model tokenization.

Five outer group folds provide evaluation predictions. Three inner group folds select D's offsets exclusively within each outer training portion. Architecture and 10 epochs are fixed before this evaluation. AdamW: learning rate 0.001, weight decay 0.01; batch size 64; gradient clipping 1. Class weights are computed from each fitting subset as `n / (4 * class_count)`.

| Variant | Controlled change |
|---|---|
| A | Full model, weighted cross-entropy, no offsets |
| B | Remove the global mean branch only |
| C | Remove class weights only |
| D | Reuse A probabilities with inner-OOF-selected log offsets |

D fixes the Negative offset at zero and searches 343 coarse plus 125 refinement candidates per outer fold. There are **150 head fits** and **2,340 inner candidate scores**. D does not fit additional heads.

This is retrospective cross-validation of a historically developed architecture, not a pristine new holdout. Outer evaluation scores are not used to retune the current run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

cwd = Path.cwd().resolve()
candidates = [base / suffix for base in (cwd, *cwd.parents) for suffix in ('', 'group project')]
project = next(base for base in candidates if (base / 'models/kun/run_experiments.py').is_file())
records = project / 'runs/kun/abcd_train_only_20261121'

In [ ]:
display(pd.read_csv(records / 'summary_metrics.csv'))

,variant,accuracy,macro_precision,macro_recall,pooled_macro_f1,fold_mean_f1,fold_sd_f1,Irrelevant_precision,Irrelevant_recall,Negative_precision,Negative_recall,delta_vs_A
0,A,0.4179,0.4016,0.4006,0.4006,0.3958,0.0197,0.2232,0.2358,0.4919,0.5230,0.0000
1,B,0.4146,0.3878,0.3921,0.3892,0.3830,0.0189,0.2093,0.1698,0.4815,0.5230,-0.0114
2,C,0.4399,0.3959,0.4057,0.3872,0.3831,0.0341,0.2258,0.0660,0.4581,0.5977,-0.0133
3,D,0.4162,0.4184,0.4011,0.4035,0.3966,0.0242,0.2206,0.2830,0.4700,0.5402,0.0030


In [ ]:
display(pd.read_csv(records / 'fold_metrics.csv'))

,outer_fold,variant,n,accuracy,macro_f1
0,1,A,119,0.420168,0.401772
1,1,B,119,0.386555,0.363953
2,1,C,119,0.436975,0.401902
3,1,D,119,0.436975,0.420639
4,2,A,120,0.441667,0.412708
5,2,B,120,0.416667,0.377135
6,2,C,120,0.441667,0.364928
7,2,D,120,0.441667,0.405071
8,3,A,117,0.435897,0.410035
9,3,B,117,0.410256,0.373929


In [ ]:
display(pd.read_csv(records / 'offset_summary.csv'))

,fold,Negative,Positive,Irrelevant,Neutral,inner_selected_f1,outer_A_f1,outer_D_f1
0,1,0.0,0.00,0.00,-0.40,0.406635,0.401772,0.420639
1,2,0.0,0.00,0.00,-0.60,0.394148,0.412708,0.405071
2,3,0.0,-0.05,0.10,-0.05,0.436045,0.410035,0.409513
3,4,0.0,0.10,0.15,-0.60,0.430718,0.389920,0.389783
4,5,0.0,-0.15,-0.15,-0.20,0.439898,0.364325,0.358047


In [ ]:
display(pd.read_csv(records / 'seed_summary.csv'))

,variant,seed,pooled_macro_f1,accuracy
0,A,101,0.395549,0.406091
1,A,102,0.399870,0.423012
2,A,103,0.393215,0.409475
3,A,104,0.392694,0.414552
4,A,105,0.410853,0.426396
5,B,101,0.380682,0.401015
6,B,102,0.402475,0.424704
7,B,103,0.402053,0.426396
8,B,104,0.370048,0.395939
9,B,105,0.396722,0.419628


In [ ]:
display(pd.read_csv(records / 'paired_error_changes.csv'))

,variant,changed,corrected_vs_A,damaged_vs_A,net_correct
0,B,51,17,19,-2
1,C,97,33,20,13
2,D,57,12,13,-1


In [ ]:
display(Image(filename=str(records / 'figures/summary.png')))

![summary.png](../../runs/kun/abcd_train_only_20261121/figures/summary.png)

In [ ]:
display(Image(filename=str(records / 'figures/confusion.png')))

![confusion.png](../../runs/kun/abcd_train_only_20261121/figures/confusion.png)

In [ ]:
display(Image(filename=str(records / 'figures/fold_comparison.png')))

![fold_comparison.png](../../runs/kun/abcd_train_only_20261121/figures/fold_comparison.png)

### Contributor-saved audit
The original checkpoint and per-row verification is recorded below.

In [ ]:
display(json.loads((records / 'author_audit.json').read_text(encoding='utf-8')))

In [ ]:
# Supply the contributor's original OOF file to repeat the row-level check.
oof_path = records / 'oof_predictions.csv'
if oof_path.is_file():
    from sklearn.metrics import accuracy_score, f1_score
    from math import isclose

    predictions = pd.read_csv(oof_path)
    assert len(predictions) == 591 and predictions.row_id.is_unique
    summary = pd.read_csv(records / 'summary_metrics.csv').set_index('variant')
    for variant in 'ABCD':
        scores = {
            'accuracy': accuracy_score(predictions.truth, predictions[variant]),
            'pooled_macro_f1': f1_score(predictions.truth, predictions[variant], average='macro'),
        }
        for metric, score in scores.items():
            # The source notebook displayed summary metrics to four decimal places.
            assert isclose(score, summary.loc[variant, metric], rel_tol=0, abs_tol=0.0000500001)
    print('Verified A/B/C/D accuracy and macro-F1 against 591 OOF predictions.')
else:
    print('Row-level check skipped: the original OOF file was not supplied.')

## 8. Interpretation and limitations
- A exceeds B in pooled Macro-F1 by **1.14 percentage points**, with wins in 3/5 folds. This supports the global branch under the fixed configuration, not uniform gains across folds.
- A exceeds C by **1.33 points**. Although C has higher accuracy, Irrelevant recall drops from **23.58% to 6.60%**.
- D improves pooled Macro-F1 by only **0.30 points**, winning in **1/5 folds**. Its gain is small and inconsistent. D gets 246 predictions correct versus A's 247.
- No statistical-significance claim is made from five overlapping-training folds.

The historical 0.5271 Macro-F1 / 54.16% accuracy came from the 397-row development evaluation used in earlier selection; it must not be presented as an untouched 372-row test score. Earlier reconstructed 372-row results were approximately 0.5060 / 52.15%, which also do not erase historical development exposure. Historical values are context only and are not recalculated here. Current folds use smaller fitting sets and different evaluation rows, so score differences cannot be attributed solely to leakage.

**Contribution:** Huakun Huang's local/global ELMo classifier, equal-weight ensemble, controlled ablations, inner-only offset selection and evaluation analysis.